<a href="https://colab.research.google.com/github/valliansayoga/ey-data-challenge-2025/blob/master/EY2025_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [21]:
to_drop = ["Latitude", "Longitude", "datetime"]
target = "UHI Index"

df = pd.read_csv("Train_Final.csv").drop(to_drop, axis=1, errors="ignore")

# # Uncomment if used
df.drop(columns=df.columns[df.columns.str.contains("count", regex=False)], inplace=True)


df.head()

,UHI Index,nearest_building_distance,red,green,blue,nir,ndvi_median,temp_median
0,1.030289,19.079136,0.124005,0.112592,0.093287,0.196825,0.226974,38.431539
1,1.030289,19.233293,0.124005,0.112592,0.093287,0.196825,0.226974,38.431539
2,1.023798,20.268009,0.071398,0.073570,0.052175,0.197622,0.469203,37.785534
3,1.023798,20.968705,0.071398,0.073570,0.052175,0.197622,0.469203,37.785534
4,1.021634,16.324876,0.071398,0.073570,0.052175,0.197622,0.469203,37.785534


In [22]:
def create_train(df_features, scaler, train_size=0.8):
    print("Removing duplicates...")
    rows_before = df_features.shape[0]
    check_dupl = df_features.columns[1:]
    df_features = df_features.drop_duplicates(subset=check_dupl, keep='first')
    rows_after = df_features.shape[0]
    print(f"Removed {rows_before-rows_after} duplicate rows!")

    X = df_features.drop(target, axis=1)
    y = df_features[target]

    print("Scaling...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0, train_size=train_size)
    X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)
    print("Done")
    return X_train, X_test, y_train, y_test, scaler
scaler = StandardScaler()
X_train, X_test, y_train, y_test, scaler = create_train(df, scaler)

Removing duplicates...
Removed 348 duplicate rows!
Scaling...
Done


# Modelling

In [23]:
from sklearn.metrics import r2_score


def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    insample = r2_score(y_train, model.predict(X_train))
    outsample = r2_score(y_test, model.predict(X_test))
    return insample, outsample

In [30]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor

models = [
    {"model": RandomForestRegressor(200, random_state=0, n_jobs=-1)},
    {"model": RandomForestRegressor(250, random_state=0, n_jobs=-1)},
    {"model": RandomForestRegressor(300, random_state=0, n_jobs=-1)},
    {"model": RandomForestRegressor(150, random_state=0, n_jobs=-1)},
    {"model": ExtraTreesRegressor(n_estimators=200, random_state=0, n_jobs=-1, bootstrap=True, oob_score=True)},
    {"model": ExtraTreesRegressor(n_estimators=200, random_state=0, n_jobs=-1)},
    {"model": ExtraTreesRegressor(n_estimators=250, random_state=0, n_jobs=-1)},
    {"model": ExtraTreesRegressor(n_estimators=100, random_state=0, n_jobs=-1)},
    {"model": KNeighborsRegressor(3, n_jobs=-1)},
    {"model": KNeighborsRegressor(5, n_jobs=-1)},
    {"model": DecisionTreeRegressor(random_state=0)},
    # {"model": MLPRegressor((8, 16, 32), random_state=0)},
]

for model in models:
    insample, outsample = evaluate_model(model["model"], X_train, X_test, y_train, y_test)
    model["insample"] = insample
    model["outsample"] = outsample

results = pd.DataFrame(models).sort_values("outsample", ascending=False).reset_index(drop=True)
results

,model,insample,outsample
0,"(ExtraTreeRegressor(random_state=209652396), E...",1.000000,0.847113
1,"(ExtraTreeRegressor(random_state=209652396), E...",1.000000,0.847027
2,"(ExtraTreeRegressor(random_state=209652396), E...",1.000000,0.846915
3,"(ExtraTreeRegressor(random_state=209652396), E...",0.973821,0.807423
4,"(DecisionTreeRegressor(max_features=1.0, rando...",0.969532,0.777367
5,"(DecisionTreeRegressor(max_features=1.0, rando...",0.969618,0.777137
6,"(DecisionTreeRegressor(max_features=1.0, rando...",0.969580,0.777042
7,"(DecisionTreeRegressor(max_features=1.0, rando...",0.969198,0.776663
8,DecisionTreeRegressor(random_state=0),1.000000,0.632242
9,"KNeighborsRegressor(n_jobs=-1, n_neighbors=3)",0.858573,0.594399


In [31]:
best_model = results.iloc[0]
best_model.model

ExtraTreesRegressor(n_jobs=-1, random_state=0)

In [32]:
importance = pd.DataFrame(
    {"Features": X_train.columns, "Importance": best_model.model.feature_importances_},
).sort_values("Importance", ascending=False).reset_index(drop=True)
importance

,Features,Importance
0,temp_median,0.226750
1,ndvi_median,0.180438
2,blue,0.124058
3,nir,0.120720
4,green,0.120504
5,red,0.120196
6,nearest_building_distance,0.107335


# Predicting Submission

In [33]:
def create_submission(filename: str, model, scaler):
    to_drop = ["Latitude", "Longitude"]
    sub_df = pd.read_csv("Submission_Final.csv")
    final_df = sub_df[to_drop].copy()
    print("Predicting", sub_df.shape[0], "rows...")
    to_predict = pd.DataFrame(
        scaler.transform(sub_df.loc[:, X_train.columns]),
        columns=X_train.columns
    )

    print("Predicting...")
    final_df["UHI Index"] = model.predict(to_predict)
    final_df.to_csv(filename, index=False)
    print("Done!")
    return
create_submission("BestModel_BldngFt_NoDupsNoCount2.csv", best_model.model, scaler)

Predicting 1040 rows...
Predicting...
Done!
